# Exploratory Data Analysis (EDA): Non-Academic Labour Well-being (AT & CZ)

This notebook performs a detailed exploratory analysis on the **non-academic staff subset** of the final processed dataset (`Final_Dataset_Processed.csv`). The primary goal is to prepare the data and gain insights for subsequent modeling focused on explaining **Burnout_Score** within this specific subgroup.

**Objectives:**
1.  **Load and Filter:** Load the processed data, filter it to retain only non-academic staff, and apply necessary label encoding for interpretability.
2.  **Handle Subgroup-Specific Data:** Identify and handle variables that are null for this subgroup (e.g., academic-specific metrics), ensuring a clean and relevant dataset for analysis.
3.  **Univariate Analysis:** Understand the distribution of all relevant variables for non-academic staff, including demographics and job-specific attributes like `Job_Description_Category` and `Career_Length_HE`.
4.  **Multivariate Analysis (Correlations):** Explore relationships between key numerical variables and `Burnout_Score`.
5.  **Bivariate Analysis (Categorical vs. Numerical):** Systematically visualize how numerical variable distributions (like `Burnout_Score` and `Job_Satisfaction`) change across all relevant non-academic categories (e.g., `Job_Description_Category_Label`, `Education_Level_Label`).
6.  **Disparity Analysis:** Investigate potential differences in well-being metrics across demographic and institutional groups within the non-academic cohort.

## 1. Setup: Import Libraries

In [ ]:
# Core Libraries for Data Manipulation
import pandas as pd
import numpy as np
import math # For calculating subplot grid size

# Libraries for Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde # For smooth density lines in histograms

# Utilities
from IPython.display import display, Markdown # For displaying outputs nicely in Jupyter
import warnings # To manage warnings

# --- Configuration ---
warnings.filterwarnings('ignore') # Suppress routine warnings for cleaner output
sns.set_theme(style="whitegrid", palette="muted") # Set consistent plot theme
plt.rcParams['figure.figsize'] = (14, 6) # Default figure size
plt.rcParams['axes.titlesize'] = 16 # Title font size
plt.rcParams['axes.labelsize'] = 12 # Axis label font size
plt.rcParams['xtick.labelsize'] = 10 # X-tick label size
plt.rcParams['ytick.labelsize'] = 10 # Y-tick label size
plt.rcParams['figure.autolayout'] = True # Enable auto layout to prevent overlaps
pd.set_option('display.max_columns', None) # Show all columns in DataFrames
pd.set_option('display.width', 1000) # Adjust display width

## 2. Data Loading and Filtering for Non-Academic Staff

Load the final processed dataset and filter it to create a new DataFrame containing only non-academic staff.

In [ ]:
# Specify the path to your final processed dataset
file_path = 'https://raw.githubusercontent.com/EduardoAve/Labour-well-being/refs/heads/main/data/02_processed/Final_Dataset_Processed.csv'

try:
    # Attempt to load the dataset
    df_raw = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' loaded successfully.")
    print(f"Original dimensions: {df_raw.shape}")

    # --- Filter for Non-Academic Staff ---
    # According to the data dictionary, 'Academic/Non-academic' == 1 means Non-academic.
    df_analysis = df_raw[df_raw['Academic/Non-academic'] == 1].copy()
    print(f"\nFiltered for Non-Academic staff. New dimensions: {df_analysis.shape}")
    display(df_analysis.head())

except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Please ensure the file is in the correct directory or provide the full path.")
    df_analysis = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    df_analysis = pd.DataFrame()

## 3. Initial Data Check (Non-Academic Sample)

Verify data types, check for missing values, and drop columns that are entirely null for this subgroup (i.e., academic-specific variables).

In [ ]:
if not df_analysis.empty:
    # --- Drop Columns with All Null Values ---
    # These are variables that do not apply to the non-academic sample (e.g., teaching hours)
    all_nan_cols = df_analysis.columns[df_analysis.isnull().all()].tolist()
    if all_nan_cols:
        print(f"\nDropping columns with all null values (irrelevant for non-academics): {all_nan_cols}")
        df_analysis.drop(columns=all_nan_cols, inplace=True)
    else:
        print("\nNo columns with all null values to drop.")

    print("\n--- General Information and Data Types ---")
    df_analysis.info()

    print("\n--- Missing Values Summary ---")
    missing_values = df_analysis.isnull().sum()
    missing_percentage = (missing_values / len(df_analysis)) * 100
    missing_info = pd.DataFrame({'Count': missing_values, 'Percentage': missing_percentage})
    missing_info = missing_info[missing_info['Count'] > 0]
    if not missing_info.empty:
        print("Warning: Missing values found in the non-academic dataset:")
        display(missing_info.sort_values(by='Count', ascending=False))
    else:
        print("No missing values found in the loaded dataset.")

    # Define conceptually categorical columns (using final names from data prep)
    categorical_original_names = [
        'Country', 'Gender', 'Marital_Status', 'Cares_for_Dependents',
        'Institution_Type', 'Subject_Area', 'Contract_Duration',
        'Effort_Comparison', 'Holds_Leadership_Position', 'Policy_Influence', 'Has_Other_Job',
        'Academic/Non-academic', 'Current_Position', 'Job_Description_Category',
        'Education_Level'
    ]
    cols_to_exclude_from_numeric = ['Version'] + categorical_original_names

    print("\n--- Initial Descriptive Statistics (Numerical Variables) ---")
    numeric_cols_initial = df_analysis.select_dtypes(include=np.number).columns
    numeric_cols_for_desc = numeric_cols_initial.difference(cols_to_exclude_from_numeric, sort=False)

    if not numeric_cols_for_desc.empty:
        display(df_analysis[numeric_cols_for_desc].describe().T.round(2))
    else:
        print("No numerical columns identified for descriptive statistics.")

else:
    print("DataFrame is empty. Further analysis cannot proceed.")

## 4. Data Preparation for EDA (Labeling)

Steps:
1.  **Label Encoding:** Convert numerical categorical variables to meaningful text labels based on the data dictionary.
2.  **Create Age Groups:** Bin the 'Age' variable for grouped analysis.

### 4.1 Label Encoding for Categorical Variables

In [ ]:
if not df_analysis.empty:
    print("Applying label encoding to categorical variables...")
    # --- Define Mappings (CRITICAL: Verify these match your actual data codes and desired labels) ---
    country_map = {1: 'Austria', 2: 'Czech Republic'}
    gender_map = {1.0: 'Male', 2.0: 'Female', 3.0: 'Other'}
    marital_map = {
        1.0: 'Married/Registered Partnership', 2.0: 'In a relationship (unmarried)',
        3.0: 'Single', 4.0: 'Divorced', 5.0: 'Widowed', 6.0: 'Other_Marital'
    }
    care_map = {
        1.0: 'No', 2.0: 'Care for underage children', 3.0: 'Care for dependent relatives',
        4.0: 'Combination Care'
    }
    institution_type_map = {
        1.0: 'CZ: Public HEI', 2.0: 'CZ: Private HEI', 3.0: 'CZ: State HEI',
        4.0: 'AT: Public University', 5.0: 'AT: Private University/College',
        6.0: 'AT: University of Applied Sciences',
        7.0: 'AT: Public Uni College Teacher Ed', 8.0: 'AT: Private Uni College Teacher Ed'
    }
    subject_area_map = {
        1.0: 'Natural sciences', 2.0: 'Technical sciences', 3.0: 'Agricultural/forestry/veterinary',
        4.0: 'Healthcare/medical/pharmaceutical', 5.0: 'Humanities/social sciences',
        6.0: 'Economic sciences', 7.0: 'Law', 8.0: 'Pedagogy/teacher training',
        9.0: 'Culture/art', 10.0: 'Sport sciences', 11.0: 'Unspecified/cannot be categorised',
        12.0: 'Security/defence/Military', 13.0: 'Other_Faculty'
    }
    contract_duration_map = {
        1.0: 'Permanent/Continuous (CZ/AT)', 2.0: 'Fixed-term (permanent prospects) (CZ/AT)',
        3.0: 'Fixed-term (no permanent prospects) (CZ/AT)', 4.0: 'Casual/hourly (CZ/AT)',
        5.0: 'Fixed-term (unspecified prospects AT?)', 6.0: 'Permanent (tenured AT?)',
        7.0: 'Other_Contract'
    }
    effort_comparison_map = {1.0: 'Equal', 2.0: 'Less', 3.0: 'More'}
    holds_leadership_map = {
        1.0: 'No', 2.0: 'Yes (Institution/Faculty/Dept)', 3.0: 'Yes (Research Team)',
        4.0: 'Combination Leadership'
    }
    policy_influence_map = {1.0: '1: Not influential', 2.0: '2', 3.0: '3', 4.0: '4', 5.0: '5: Very influential'}
    academic_non_academic_map = {1: 'Non-academic', 2: 'Academic'}
    current_position_map = {
        2.0: 'Lecturer (CZ/AT Lector)', 3.0: 'Assistant (CZ)', 4.0: 'Assistant professor (CZ)',
        5.0: 'Docent (CZ)', 6.0: 'Professor (CZ)', 7.0: 'Researcher (CZ)',
        8.0: 'Externist (CZ)', 9.0: 'Researcher & Academic (CZ)',
        10.0: 'Postdoc Assistant (AT)', 11.0: 'Assistant Prof (AT)', 12.0: 'Associate Prof (AT)',
        13.0: 'University Prof (AT)', 14.0: 'Senior Scientist/Lecturer (AT)', 15.0: 'Project Staff (AT)',
        16.0: 'Other Position (AT)', 17.0: 'Student Assistant (AT)',
        1.0: 'Non-academic Role (Generic)', 0.0: 'Unknown/NA Position'
    }
    job_description_map = {
        1.0: 'Dept/manager assistant', 2.0: 'Lab technician', 3.0: 'Librarian/archivist',
        4.0: 'Facility management', 5.0: 'ICT', 6.0: 'Student support',
        7.0: 'Economics/finance/HR', 8.0: 'Project management', 9.0: 'Legal/control',
        10.0: 'Marketing/PR', 11.0: 'Science/knowledge transfer', 12.0: 'Foreign affairs',
        13.0: 'Other administrative', 14.0: 'Other/Combination Admin'
    }
    education_level_map = {
        1.0: 'Elementary', 2.0: 'Apprenticeship', 3.0: 'Vocational/Commercial School',
        4.0: 'High school/Secondary', 5.0: 'Higher professional school (CZ)',
        6.0: 'Bachelor', 7.0: 'Master', 8.0: 'Doctoral/PhD'
    }
    has_other_job_map = {
        1.0: 'No', 2.0: 'Yes (Public Sector)', 3.0: 'Yes (Private Sector)',
        4.0: 'Yes (Non-profit)', 5.0: 'Yes (Self-employed)',
        6.0: 'Yes (Multiple Areas)', 7.0: 'Yes (Other_Job)'
    }

    mappings_to_apply = {
        'Country': country_map,
        'Gender': gender_map,
        'Marital_Status': marital_map,
        'Cares_for_Dependents': care_map,
        'Institution_Type': institution_type_map,
        'Subject_Area': subject_area_map,
        'Contract_Duration': contract_duration_map,
        'Effort_Comparison': effort_comparison_map,
        'Holds_Leadership_Position': holds_leadership_map,
        'Policy_Influence': policy_influence_map,
        'Academic/Non-academic': academic_non_academic_map,
        'Current_Position': current_position_map,
        'Job_Description_Category': job_description_map,
        'Education_Level': education_level_map,
        'Has_Other_Job': has_other_job_map
    }

    labeled_cols = []

    for col, mapping in mappings_to_apply.items():
        if col in df_analysis.columns:
            label_col = f"{col}_Label"
            df_analysis[label_col] = df_analysis[col].map(mapping)
            is_ordered = (col == 'Policy_Influence' or col == 'Effort_Comparison' or col == 'Education_Level')
            try:
                all_categories = sorted(list(set(mapping.values()))) if not is_ordered else list(mapping.values())
                df_analysis[label_col] = pd.Categorical(df_analysis[label_col], categories=all_categories, ordered=is_ordered)
                print(f"  - Labeled column '{label_col}' created for '{col}'. Type: {'Ordinal' if is_ordered else 'Nominal'}")
                labeled_cols.append(label_col)
            except Exception as e:
                print(f"  - Warning: Could not convert '{label_col}' to categorical. Error: {e}")
                if label_col in df_analysis.columns:
                    df_analysis[label_col] = df_analysis[label_col].astype('object')
        else:
            # This is now expected for academic-only variables, so no warning is needed.
            pass

    print("\nLabel encoding completed.")

    if labeled_cols:
        display(df_analysis[labeled_cols].head())

else:
    print("DataFrame is empty, label encoding skipped.")

### 4.2 Create Age Groups

In [ ]:
if not df_analysis.empty:
    age_bins = [0, 29.9, 39.9, 49.9, 59.9, np.inf]
    age_labels = ['<30', '30-39', '40-49', '50-59', '60+']
    if 'Age' in df_analysis.columns:
        df_analysis['Age_Group'] = pd.cut(df_analysis['Age'], bins=age_bins, labels=age_labels, right=True)
        df_analysis['Age_Group'] = pd.Categorical(df_analysis['Age_Group'], categories=age_labels, ordered=True)
        print("'Age_Group' column created successfully.")
        if 'labeled_cols' in locals() and 'Age_Group' not in labeled_cols:
            labeled_cols.append('Age_Group')
    else:
        print("Warning: 'Age' column not found, could not create 'Age_Group'.")
else:
    print("Empty DataFrame, age group creation skipped.")

## 5. Exploratory Data Analysis (EDA) - Non-Academic Staff

Analyze distributions, relationships, and potential disparities in the non-academic staff data (`df_analysis`).

### 5.1 Univariate Analysis (Non-Academic Staff)

Examine the distribution of each variable for the non-academic cohort.

In [ ]:
if not df_analysis.empty:
    # --- Identify final numeric and categorical columns for analysis ---
    all_numeric_df_cols = df_analysis.select_dtypes(include=np.number).columns
    # Exclude original categorical columns that are stored as numbers
    numeric_cols_for_analysis = all_numeric_df_cols.difference(categorical_original_names, sort=False)
    # Get all labeled columns and age group
    categorical_cols_for_analysis = [col for col in df_analysis.columns if col.endswith('_Label') or col == 'Age_Group']

    # --- Numerical Descriptive Statistics ---
    display(Markdown("#### Descriptive Statistics (Numerical Variables - Non-Academic Staff)"))
    if not numeric_cols_for_analysis.empty:
        display(df_analysis[numeric_cols_for_analysis].describe().T.round(2))
    else:
        print("No numerical columns found for descriptive statistics.")

    # --- Categorical Frequencies ---
    display(Markdown("#### Frequencies (Categorical Variables - Labeled - Non-Academic Staff)"))
    if categorical_cols_for_analysis:
        for col in categorical_cols_for_analysis:
            if col in df_analysis.columns:
                display_col_name = col.replace('_Label','').replace('_',' ')
                display(Markdown(f"##### {display_col_name}"))
                freq_table = df_analysis[col].value_counts(dropna=False).to_frame(name="Frequency")
                freq_table['Percentage'] = (df_analysis[col].value_counts(normalize=True, dropna=False) * 100).round(2)
                display(freq_table)
            else:
                print(f"Warning: Column {col} not found for frequency count.")
    else:
        print("No labeled categorical columns found for frequency counts.")

else:
    print("DataFrame is empty, univariate statistics skipped.")

In [ ]:
if not df_analysis.empty:
    display(Markdown("#### Univariate Visualizations (Non-Academic Staff)"))
    uni_palette = "viridis"
    fig_size_uni = (14, 5)

    # --- Plots for Numerical Variables ---
    if not numeric_cols_for_analysis.empty:
        display(Markdown("##### Numerical Distributions (Histograms & Boxplots - Non-Academic Staff)"))
        for col in numeric_cols_for_analysis:
            clean_col_name = col.replace('_', ' ')
            try:
                if df_analysis[col].notna().sum() < 2: continue
                fig, axes = plt.subplots(1, 2, figsize=fig_size_uni)
                fig.suptitle(f'Distribution of {clean_col_name} (Non-Academic Staff)', fontsize=16, y=1.03)
                sns.histplot(df_analysis[col], kde=True, ax=axes[0], bins=30, color=sns.color_palette(uni_palette, 2)[0])
                axes[0].set_title('Histogram & Density'); axes[0].set_xlabel(clean_col_name); axes[0].set_ylabel('Frequency / Density')
                sns.boxplot(x=df_analysis[col], ax=axes[1], color=sns.color_palette(uni_palette, 2)[1])
                axes[1].set_title('Boxplot'); axes[1].set_xlabel(clean_col_name)
                plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()
            except Exception as e:
                print(f"Could not plot numerical variable {col}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
    else:
        print("No numerical columns for plotting.")

    # --- Plots for Categorical Variables ---
    if categorical_cols_for_analysis:
        display(Markdown("##### Categorical Distributions (Bar Charts - Non-Academic Staff)"))
        for col in categorical_cols_for_analysis:
            if col in df_analysis.columns:
                clean_col_name = col.replace('_Label','').replace('_',' ')
                try:
                    num_categories = df_analysis[col].nunique(dropna=False)
                    if df_analysis[col].notna().any() and num_categories > 0:
                        dynamic_height = max(5, num_categories * 0.45)
                        plt.figure(figsize=(10, dynamic_height))
                        order = df_analysis[col].value_counts(dropna=False).index
                        ax = sns.countplot(y=df_analysis[col], order=order, palette=uni_palette)
                        ax.set_title(f'Distribution of {clean_col_name} (Non-Academic Staff)'); ax.set_xlabel('Frequency Count'); ax.set_ylabel('')
                        plt.yticks(fontsize=9); plt.tight_layout(); plt.show()
                    else: pass
                except Exception as e:
                    print(f"Could not plot categorical variable {col}. Error: {e}")
                    if plt.gcf().get_axes(): plt.close()
            else:
                print(f"Warning: Column {col} not found for plotting.")
    else:
        print("No labeled categorical columns for plotting.")
else:
    print("DataFrame is empty, univariate visualizations skipped.")

### 5.2 Multivariate Analysis (Correlations & Pairplots - Non-Academic Staff)

Explore relationships between numerical variables in the non-academic sample.

#### 5.2.1 Scatter Plot Matrices (Pairplots - Non-Academic Staff)

In [ ]:
if not df_analysis.empty:
    display(Markdown("##### Scatter Plot Matrices (Pairplots - Non-Academic Staff, Hue by Country)"))

    all_numeric_vars_for_plots = numeric_cols_for_analysis

    # Note: Academic-specific variables like 'Academic_Resources' might be less relevant but are kept for comparison.
    # Non-academic specific variables like 'Career_Length_HE' are now included.
    pairplot1_vars_candidates = [
        'Age', 'Burnout_Score', 'Job_Satisfaction', 'Salary/hour', 'Avg_Work_Hours_HE',
        'Perceived_Autonomy', 'Performance_Pressure', 'Quality_Leadership',
        'Sense_Community', 'Career_Length_HE'
    ]
    pairplot1_vars = [var for var in pairplot1_vars_candidates if var in all_numeric_vars_for_plots]

    pairplot2_vars_candidates = [col for col in all_numeric_vars_for_plots if col.startswith('VB_')] + ['Burnout_Score']
    pairplot2_vars = [var for var in pairplot2_vars_candidates if var in all_numeric_vars_for_plots]

    pairplot3_vars_candidates = [col for col in all_numeric_vars_for_plots if col.startswith('WM_')] + ['Burnout_Score']
    pairplot3_vars = [var for var in pairplot3_vars_candidates if var in all_numeric_vars_for_plots]

    def generate_pairplot_general(data, vars_list, title_suffix, hue_col='Country_Label'):
        if len(vars_list) > 1:
            print(f"Generating pairplot for: {vars_list} ({len(vars_list)} variables), hue by {hue_col}")
            if hue_col not in data.columns:
                print(f"Warning: Hue column '{hue_col}' not found. Plotting without hue.")
                hue_col = None

            plot_data_cols = vars_list + ([hue_col] if hue_col else [])
            pairplot_data = data[plot_data_cols].copy()

            try:
                g = sns.pairplot(pairplot_data.dropna(subset=vars_list),
                                 hue=hue_col,
                                 diag_kind='kde',
                                 plot_kws={'alpha': 0.4, 's': 30, 'edgecolor': None},
                                 height=1.8)

                plt.suptitle(f'Scatter Plot Matrix: {title_suffix} (Non-Academic)', y=1.02, fontsize=14)
                g.fig.tight_layout(rect=[0, 0, 1, 0.98])
                plt.show()
            except Exception as e:
                print(f"Could not generate pairplot for {title_suffix}. Error: {e}")
                if plt.gcf().get_axes(): plt.close()
        else:
            print(f"Skipping pairplot for {title_suffix}: Not enough variables ({len(vars_list)} found).")

    display(Markdown("###### Pairplot 1: Work Conditions & Demographics vs. Burnout"))
    generate_pairplot_general(df_analysis, pairplot1_vars, "Work Conditions & Demographics vs. Burnout")
    display(Markdown("###### Pairplot 2: Work Attitudes (VB_) vs. Burnout"))
    generate_pairplot_general(df_analysis, pairplot2_vars, "Work Attitudes (VB_) vs. Burnout")
    display(Markdown("###### Pairplot 3: Work Motivations (WM_) vs. Burnout"))
    generate_pairplot_general(df_analysis, pairplot3_vars, "Work Motivations (WM_) vs. Burnout")

else:
    print("DataFrame is empty, pairplots skipped.")

#### 5.2.2 Full Correlation Matrix (Pearson - Non-Academic Staff)

Visualize the linear relationships between all numerical variables for non-academics.

In [ ]:
if not df_analysis.empty:
    display(Markdown("##### Full Correlation Matrix (Pearson - Non-Academic Staff)"))

    numeric_cols_for_corr = numeric_cols_for_analysis

    if not numeric_cols_for_corr.empty and len(numeric_cols_for_corr) > 1:
        correlation_matrix_full = df_analysis[numeric_cols_for_corr].corr(method='pearson')
        mask = np.triu(np.ones_like(correlation_matrix_full, dtype=bool))
        plt.figure(figsize=(max(12, len(numeric_cols_for_corr)*0.6), max(10, len(numeric_cols_for_corr)*0.5)))
        sns.heatmap(correlation_matrix_full,
                    mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
                    linewidths=.5, cbar_kws={"shrink": .7}, annot=False, fmt=".2f")
        plt.title('Full Correlation Matrix (Pearson - Non-Academic Staff)', fontsize=16)
        plt.xticks(rotation=60, ha='right', fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        plt.tight_layout()
        plt.show()
    elif len(numeric_cols_for_corr) <= 1:
        print("Not enough quantitative columns (>1) to calculate correlation matrix.")
    else:
        print("No quantitative columns found for correlation analysis.")
else:
    print("DataFrame is empty, full correlation analysis skipped.")

#### 5.2.3 Grouped Correlation Matrices (Non-Academic Staff)

Examine correlations within domains and between predictors and `Burnout_Score`.

In [ ]:
# Define the heatmap plotting function here, before it's called
def plot_correlation_heatmap(corr_matrix, title):
    """Helper function to plot a correlation heatmap."""
    if corr_matrix.empty or corr_matrix.shape[0] < 1 or corr_matrix.shape[1] < 1:
        print(f"Skipping heatmap for '{title}': Not enough variables or empty matrix.")
        return
    mask = None
    if corr_matrix.shape[0] == corr_matrix.shape[1] and corr_matrix.shape[0] > 1:
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    annot_size = 8 if max(corr_matrix.shape) < 15 else 7
    plt.figure(figsize=(max(8, corr_matrix.shape[1]*0.8), max(6, corr_matrix.shape[0]*0.6)))
    sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', vmax=1, vmin=-1, center=0,
                linewidths=.5, cbar_kws={"shrink": .7}, annot=True, fmt=".2f", annot_kws={"size": annot_size})
    plt.title(title, fontsize=14); plt.xticks(rotation=45, ha='right', fontsize=9); plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout(); plt.show()

if not df_analysis.empty:
    display(Markdown("##### Grouped Correlation Matrices (Non-Academic Staff)"))
    all_numeric_vars_corr = numeric_cols_for_analysis

    # Adjusted variable groups for non-academic staff
    demographic_vars_num = [v for v in ['Age', 'Career_Length_HE'] if v in all_numeric_vars_corr]
    work_conditions_vars_num = [v for v in ['Avg_Work_Hours_HE', 'Salary/hour', 'Salary effort/hour', 'Avg_Work_Hours_Other'] if v in all_numeric_vars_corr]
    subjective_perception_vars = [v for v in ['Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy', 'Quality_Leadership', 'Sense_Community'] if v in all_numeric_vars_corr]
    motivation_vars = [v for v in all_numeric_vars_corr if v.startswith('WM_')]
    attitudes_vars = [v for v in all_numeric_vars_corr if v.startswith('VB_')]
    wellbeing_outcomes = [v for v in ['Job_Satisfaction', 'Burnout_Score', 'Vulnerability'] if v in all_numeric_vars_corr]

    if 'correlation_matrix_full' not in locals() or correlation_matrix_full.empty:
        if not all_numeric_vars_corr.empty and len(all_numeric_vars_corr) > 1:
            correlation_matrix_full = df_analysis[all_numeric_vars_corr].corr(method='pearson')
        else:
            correlation_matrix_full = pd.DataFrame()

    if not correlation_matrix_full.empty:
        groups_to_plot_intra = {
            "Demographics": demographic_vars_num,
            "Work Conditions (Numeric)": work_conditions_vars_num,
            "Subjective Perceptions": subjective_perception_vars,
            "Motivations (WM_)": motivation_vars,
            "Attitudes (VB_)": attitudes_vars,
            "Well-being Outcomes": wellbeing_outcomes
        }
        for group_name, group_vars in groups_to_plot_intra.items():
            valid_group_vars = [var for var in group_vars if var in all_numeric_vars_corr]
            if len(valid_group_vars) > 1:
                group_corr = correlation_matrix_full.loc[valid_group_vars, valid_group_vars]
                plot_correlation_heatmap(group_corr, f'Intra-Group Correlation: {group_name} (Non-Academic)')
            else:
                print(f"Skipping intra-group heatmap for '{group_name}': Not enough valid variables.")

        key_outcome_burnout = ['Burnout_Score']
        valid_key_outcome_burnout = [var for var in key_outcome_burnout if var in all_numeric_vars_corr]
        predictor_groups_numeric = {
            "Demographics & Work Conditions": demographic_vars_num + work_conditions_vars_num,
            "Subjective Perceptions": subjective_perception_vars,
            "Motivations (WM_)": motivation_vars,
            "Attitudes (VB_)": attitudes_vars
        }
        if valid_key_outcome_burnout:
            display(Markdown(f"#### Correlations between Predictor Groups and {valid_key_outcome_burnout[0]} (Non-Academic)"))
            for group_name, pred_vars in predictor_groups_numeric.items():
                valid_pred_vars = [var for var in pred_vars if var in all_numeric_vars_corr]
                if valid_pred_vars and valid_key_outcome_burnout[0] in correlation_matrix_full.columns:
                    cross_corr = correlation_matrix_full.loc[valid_pred_vars, valid_key_outcome_burnout]
                    if not cross_corr.empty: plot_correlation_heatmap(cross_corr, f'Correlations: {group_name} vs. {valid_key_outcome_burnout[0]} (Non-Academic)')
                    else: print(f"No valid correlations for '{group_name}' vs. {valid_key_outcome_burnout[0]}.")
                else: print(f"Skipping cross-correlation for '{group_name}': Invalid predictors or outcome.")
        else: print(f"Key outcome variable ({key_outcome_burnout[0]}) not found or not numeric.")
    else: print("Full correlation matrix is empty.")
else: print("DataFrame is empty, grouped correlation analysis skipped.")

### 5.3 Bivariate Analysis: Categorical vs. Numerical Variables (Non-Academic Staff)

Visualize distributions of numerical variables across non-academic staff categories.

In [ ]:
if 'df_analysis' in locals() and not df_analysis.empty:
    numeric_cols_for_biv_plots = numeric_cols_for_analysis
    categorical_cols_for_biv_plots = categorical_cols_for_analysis
    biv_palette = "pastel"

    if not numeric_cols_for_biv_plots.empty and categorical_cols_for_biv_plots:
        display(Markdown("##### Comparison of Numerical Distributions by Category (Non-Academic Staff)"))
        for cat_col in categorical_cols_for_biv_plots:
            if cat_col in df_analysis.columns:
                clean_cat_col_name = cat_col.replace('_Label','').replace('_',' ')
                display(Markdown(f"###### Comparisons by: {clean_cat_col_name}"))
                for num_col in numeric_cols_for_biv_plots:
                    clean_num_col_name = num_col.replace('_', ' ')
                    try:
                        num_categories = df_analysis[cat_col].nunique(dropna=False)
                        if df_analysis[[cat_col, num_col]].dropna().empty or num_categories == 0: continue
                        max_label_len = 0
                        if df_analysis[cat_col].dtype == 'category' and df_analysis[cat_col].cat.categories.inferred_type == 'string':
                           max_label_len = df_analysis[cat_col].cat.categories.str.len().max()
                        elif df_analysis[cat_col].dtype == 'object' and df_analysis[cat_col].dropna().apply(lambda x: isinstance(x, str)).all():
                           max_label_len = df_analysis[cat_col].str.len().max()
                        base_width_per_cat = 0.8
                        if max_label_len > 15: base_width_per_cat = max_label_len * 0.08
                        fig_width = max(10, num_categories * base_width_per_cat)
                        plt.figure(figsize=(fig_width, 6))
                        plot_order = None
                        if isinstance(df_analysis[cat_col].dtype, pd.CategoricalDtype):
                            if df_analysis[cat_col].cat.ordered: plot_order = df_analysis[cat_col].cat.categories.tolist()
                            elif num_categories < 15: plot_order = df_analysis[cat_col].value_counts().index
                        ax = sns.violinplot(x=cat_col, y=num_col, data=df_analysis, palette=biv_palette, cut=0, inner='quartile', order=plot_order)
                        ax.set_title(f'{clean_num_col_name} by {clean_cat_col_name} (Non-Academic)'); ax.set_xlabel(clean_cat_col_name); ax.set_ylabel(clean_num_col_name)
                        if num_categories > 4 or max_label_len > 10: plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                        else: plt.setp(ax.get_xticklabels(), rotation=0)
                        plt.tight_layout(); plt.show()
                    except Exception as e:
                        print(f"Could not plot {num_col} by {cat_col}. Error: {e}")
                        if plt.gcf().get_axes(): plt.close()
            else: print(f"Warning: Categorical column {cat_col} not found.")
    else: print("Not enough numerical or categorical variables for bivariate analysis.")
else: print("DataFrame is empty, Bivariate Categorical vs Numerical analysis skipped.")

### 5.4 Disparity Analysis (Compact Subplots - Non-Academic Staff)

Explore how key well-being variables differ across specific demographic and job-related groups within the non-academic cohort.

In [ ]:
if 'df_analysis' in locals() and not df_analysis.empty:
    grouping_vars_labels = categorical_cols_for_analysis
    target_vars_grouped_candidates = [
        'Burnout_Score', 'Job_Satisfaction', 'Perceived_Autonomy', 'Performance_Pressure',
        'Quality_Leadership', 'WM_Intrinsic_Motivation', 'VB_Emotional_Distancing', 'Salary/hour'
    ]
    valid_target_vars_grouped = [var for var in target_vars_grouped_candidates if var in numeric_cols_for_analysis]
    print(f"Key variables selected for compact disparity analysis: {valid_target_vars_grouped}")
    grouped_palette = "muted"; num_targets = len(valid_target_vars_grouped)

    if grouping_vars_labels and valid_target_vars_grouped:
        for group_var in grouping_vars_labels:
            if group_var in df_analysis.columns and df_analysis[group_var].notna().any() and df_analysis[group_var].nunique() > 1:
                clean_group_var_name = group_var.replace('_Label','').replace('_',' ')
                display(Markdown(f"#### Disparity Analysis by: {clean_group_var_name} (Non-Academic Staff)"))
                try:
                    use_observed = pd.__version__ >= '1.5.0' and pd.api.types.is_categorical_dtype(df_analysis[group_var])
                    grouped_stats = df_analysis.groupby(group_var, observed=use_observed)[valid_target_vars_grouped].agg(['mean', 'median'])
                    display(grouped_stats.round(2))
                except Exception as e: print(f"Could not calculate grouped stats for {group_var}. Error: {e}"); continue

                num_categories = df_analysis[group_var].nunique(dropna=False)
                if num_categories == 0: continue
                ncols = 2 if num_targets > 1 else 1; nrows = math.ceil(num_targets / ncols)
                max_cat_label_len = 0
                if df_analysis[group_var].dtype == 'category' and df_analysis[group_var].cat.categories.inferred_type == 'string':
                   max_cat_label_len = df_analysis[group_var].cat.categories.str.len().max()
                elif df_analysis[group_var].dtype == 'object' and df_analysis[group_var].dropna().apply(lambda x: isinstance(x, str)).all():
                   max_cat_label_len = df_analysis[group_var].str.len().max()
                base_width_per_cat_subplot = 0.8
                if max_cat_label_len > 15: base_width_per_cat_subplot = max_cat_label_len * 0.07
                fig_height_subplot = 5 * nrows; fig_width_subplot = max(10, num_categories * base_width_per_cat_subplot) * ncols
                fig, axes = plt.subplots(nrows, ncols, figsize=(fig_width_subplot, fig_height_subplot), squeeze=False); axes = axes.flatten()
                plot_order = None
                if isinstance(df_analysis[group_var].dtype, pd.CategoricalDtype):
                    if df_analysis[group_var].cat.ordered: plot_order = df_analysis[group_var].cat.categories.tolist()
                    elif num_categories < 15: plot_order = df_analysis[group_var].value_counts().index

                for i, target_var in enumerate(valid_target_vars_grouped):
                    ax = axes[i]; clean_target_var_name = target_var.replace('_',' ')
                    try:
                        sns.violinplot(x=group_var, y=target_var, data=df_analysis, palette=grouped_palette, cut=0, inner='quartile', order=plot_order, ax=ax)
                        ax.set_title(f'{clean_target_var_name}'); ax.set_xlabel(''); ax.set_ylabel(clean_target_var_name)
                        if num_categories > 4 or max_cat_label_len > 10: plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                        else: plt.setp(ax.get_xticklabels(), rotation=0)
                    except Exception as e: print(f"Could not plot {target_var} by {group_var} on subplot. Error: {e}"); ax.set_title(f'{clean_target_var_name} (Plot Error)')
                for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
                fig.suptitle(f'Distribution of Key Metrics by {clean_group_var_name} (Non-Academic)', fontsize=16, y=1.02)
                plt.tight_layout(rect=[0, 0, 1, 0.98]); plt.show()
            else: pass
    else: print("Could not perform disparity analysis: Missing grouping or target variables.")
else: print("DataFrame is empty, disparity analysis skipped.")

## 6. Preliminary EDA Conclusions and Next Steps (Non-Academic Staff)

This exploratory analysis of the non-academic staff dataset provides crucial groundwork for any subsequent modeling aimed at explaining `Burnout_Score` for this specific group.

**Key Observations (Non-Academic Staff):**
* **Variable Distributions:** The univariate analysis revealed the distributions of key predictors and outcomes. `Burnout_Score` appears to have a left skew, suggesting many non-academic employees report higher levels of burnout. The distribution of `Career_Length_HE` is also skewed, with many staff having shorter tenures.
* **Correlations & Relationships:** `Burnout_Score` shows expected negative correlations with `Job_Satisfaction`, `Quality_Leadership`, and several `VB_` scales like `Mental_Stability` and `Satisfaction_Work`. Conversely, it's positively correlated with `Performance_Pressure` and `VB_Tendency_Exertion`.
* **Country Differences:** Visualizations using `Country_Label` as a hue or grouping variable can reveal if relationships or mean levels of key variables differ between non-academic staff in Austria and the Czech Republic, which is important for deciding on multi-group modeling.
* **Group Differences (Job Roles & Education):** The disparity analyses across `Job_Description_Category_Label` and `Education_Level_Label` are particularly insightful for this subgroup. They suggest that burnout levels and job satisfaction may vary significantly depending on the type of administrative role (e.g., 'ICT', 'Student support', 'Economics/finance/HR') and the educational background of the employee.
* **Key Predictor Areas:** For non-academics, subjective perceptions of the work environment (leadership, pressure) and personal coping attitudes (VB_ scales) appear strongly correlated with burnout, similar to the academic cohort.

**Next Steps for Modeling:**
1.  **Model Specification:** Define a hypothesized model (e.g., SEM) based on theory and these EDA findings, focusing on predictors relevant to non-academic roles.
2.  **Feature Engineering/Selection:** Decide whether to include detailed categories (like `Job_Description_Category`) as dummy variables or to group them. Assess the importance of `Career_Length_HE` as a predictor.
3.  **Model Estimation:** Estimate a model using appropriate software. Given the potential for non-normal data, robust estimators should be considered.
4.  **Model Evaluation & Refinement:** Assess model fit and path coefficients, modifying the model as needed.
5.  **Multi-Group Analysis:** If country differences are prominent, conduct a multi-group analysis to compare the drivers of burnout for non-academic staff in Austria versus the Czech Republic.